<a href="https://colab.research.google.com/github/liangliang6v6/Homeworks/blob/pages/Homework4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###Task 1 (50 points):
This task involves training existing models. Download the
character level RNN at https://github.com/karpathy/char-rnn
You are required to read the documentation provided in this repository and experiment with the RNN model. This is a legacy repository; therefore, one task would be to research and use a recent version. Train the model on ‘tiny Shakespeare’ dataset available at the same location.
Create outputs of the model after training for i) 5 epochs ii) 50 epochs and iii) 500 epochs. What significant difference do you observe between the 3 outputs? Explain. Repeat the experiment with the LSTM model provided in the repository. Explain the differences and/or similarities between the results of both models.

### Answer
The code and results are listed below. The results from training both RNN and LSTM models across 5, 50, and 500 epochs reveal differences in their learning capabilities and the coherence of generated text.

Initially, both models start with incoherent outputs due to insufficient training, with words being a random mixture of characters rather than structured language. However, the LSTM already shows a slight edge in structure early on. By 50 epochs, the RNN begins to create somewhat understandable text but still significantly lags behind the LSTM, which produces more fluent and syntactically correct sequences. This improvement in LSTM outcomes demonstrates its ability to leverage its gating mechanisms to manage information flow better, allowing it to maintain long-term dependencies essential for meaningful sentence construction.

By 500 epochs, the LSTM further solidifies its superiority with coherent and structured dialogue, reflecting its capacity to learn complex patterns within the data effectively. The RNN improves with more training but struggles with inconsistencies, highlighting its limitation in retaining long-term contextual information due to vanishing gradient issues inherent to its architecture. The lower loss values and higher textual coherence of the LSTM underscore its adeptness at handling sequential data, validating the architectural enhancements that make it favorable for text generation tasks. These experiments reinforce the well-established preference for LSTMs in tasks requiring processing of long sequences, like language modeling and text generation.

In [16]:
import urllib.request

# Download the dataset
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
file_path = "tinyshakespeare.txt"
urllib.request.urlretrieve(url, file_path)

with open(file_path, "r") as f:
    text = f.read()

print("Dataset Sample:\n")
print(text[:1000])

# Character-to-index mapping
chars = sorted(list(set(text)))
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}
vocab_size = len(chars)

# Encode the entire dataset
encoded_text = np.array([char_to_idx[ch] for ch in text])

print(f"Unique characters: {''.join(chars)}")


Dataset Sample:

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thir

In [1]:
!nvidia-smi

Tue Feb 25 04:55:02 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [26]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import torch.nn.functional as F

# use gpu
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# RNN model
class CharRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(CharRNN, self).__init__()
        self.hidden_size = hidden_size
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x, hidden):
        out, hidden = self.rnn(x, hidden)
        out = self.fc(out)
        return out, hidden

    def init_hidden(self, batch_size):
        return torch.zeros(1, batch_size, self.hidden_size, device=device)

# train
def train(rnn, encoded_text, vocab_size, seq_length, num_epochs, learning_rate, batch_size):
    optimizer = optim.Adam(rnn.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss()

    # Move the model to GPU
    rnn.to(device)

    num_batches = (len(encoded_text) - 1) // (batch_size * seq_length)

    for epoch in range(num_epochs):
        hidden = rnn.init_hidden(batch_size)
        total_loss = 0

        for batch in range(num_batches):
            # Calculate start indices for the batch
            start_idx = batch * batch_size * seq_length
            end_idx = start_idx + seq_length * batch_size
            if end_idx >= len(encoded_text):
                break

            # Prepare batch data (inputs and targets)
            batch_inputs = []
            batch_targets = []
            for b in range(batch_size):
                start = start_idx + (b * seq_length)
                batch_inputs.append(encoded_text[start:start + seq_length])
                batch_targets.append(encoded_text[start + 1:start + seq_length + 1])

            inputs = torch.eye(vocab_size)[np.array(batch_inputs)].to(device)
            targets = torch.tensor(np.array(batch_targets), device=device)

            if isinstance(hidden, tuple):  # For LSTMs
                hidden = tuple([each.detach() for each in hidden])
            else:  # For simple RNNs
                hidden = hidden.detach()

            optimizer.zero_grad()
            output, hidden = rnn(inputs, hidden)
            loss = criterion(output.view(-1, vocab_size), targets.view(-1))
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f'Epoch {epoch + 1}/{num_epochs}, Loss: {total_loss/num_batches:.4f}')

def predict(model, char, h=None, top_k=None):
    char_to_idx, idx_to_char = model.char_to_idx, model.idx_to_char
    x = np.array([[char_to_idx[char]]])
    x = torch.nn.functional.one_hot(torch.tensor(x), num_classes=len(char_to_idx)).float()
    inputs = x.to(device)

    if h is None:
        h = model.init_hidden(1)
    # For LSTM, detach both hidden and cell states, for RNN just detach hidden state
    if isinstance(h, tuple):  # LSTM
        h = tuple([each.data for each in h])
    else:  # RNN
        h = h.data

    out, h = model(inputs, h)

    p = F.softmax(out, dim=-1).data
    p = p.cpu()

    if top_k is None:
        top_ch = np.arange(len(char_to_idx))
    else:
        p, top_ch = p.topk(top_k)
        top_ch = top_ch.numpy().squeeze()

    p = p.numpy().squeeze()
    char = np.random.choice(top_ch, p=p/p.sum())

    return idx_to_char[char], h

# Function to generate text
def generate_text(model, length, starting_text='The', top_k=None):
    model.eval()
    chars = [ch for ch in starting_text]
    char_to_idx, idx_to_char = model.char_to_idx, model.idx_to_char
    h = model.init_hidden(1)

    for ch in starting_text:
        char, h = predict(model, ch, h, top_k=top_k)
        chars.append(char)

    for ii in range(length):
        char, h = predict(model, chars[-1], h, top_k=top_k)
        chars.append(char)

    return ''.join(chars)

def train_and_generate(model, data, vocab_size, seq_length, learning_rate, batch_size, num_epochs, start_text='The', sample_length=200):
    # Train
    train(model, data, vocab_size, seq_length, num_epochs, learning_rate, batch_size)

    # Generate text after training
    print(f"\nGenerated text after {num_epochs} epochs:")
    print(generate_text(model, sample_length, starting_text=start_text, top_k=5))

if __name__ == "__main__":
    # load data
    with open("tinyshakespeare.txt", "r") as f:
      text = f.read()
    chars = sorted(list(set(text)))
    # mapping
    char_to_idx = {ch: i for i, ch in enumerate(chars)}
    idx_to_char = {i: ch for i, ch in enumerate(chars)}
    vocab_size = len(chars)
    # encode
    encoded_text = np.array([char_to_idx[ch] for ch in text])

    # model init
    hidden_size = 128
    vocab_size = len(chars)
    seq_length = 100
    num_epochs = 5
    batch_size = 64
    learning_rate = 0.005

    # testing with different epoch settings

    print(f"\nTraining RNN with 5 epochs:")
    model = CharRNN(input_size=vocab_size, hidden_size=hidden_size, output_size=vocab_size)
    model.char_to_idx = char_to_idx
    model.idx_to_char = idx_to_char
    model.chars = chars
    model.to(device)
    train_and_generate(model, encoded_text, vocab_size, seq_length, learning_rate, batch_size, num_epochs=5, start_text='The')


Using device: cuda

Training RNN with 5 epochs:
Epoch 1/5, Loss: 2.6663
Epoch 2/5, Loss: 2.1414
Epoch 3/5, Loss: 1.9927
Epoch 4/5, Loss: 1.9047
Epoch 5/5, Loss: 1.8466

Generated text after 5 epochs:
TheAon for thou deet of tay asle the world the born and all thin thou trence this battle.

Preas it not where inte on the much an hath thou are as and thou are a tranger a pristed than the bence. Whee in wi


In [19]:
# RNN 50 epochs
print(f"\nTraining RNN with 50 epochs:")
model = CharRNN(input_size=vocab_size, hidden_size=hidden_size, output_size=vocab_size)
model.char_to_idx = char_to_idx
model.idx_to_char = idx_to_char
model.chars = chars
model.to(device)
train_and_generate(model, encoded_text, vocab_size, seq_length, learning_rate, batch_size, num_epochs=50, start_text='The')


Training RNN with 50 epochs:
Epoch 1/50, Loss: 2.6091
Epoch 2/50, Loss: 2.1209
Epoch 3/50, Loss: 1.9807
Epoch 4/50, Loss: 1.8959
Epoch 5/50, Loss: 1.8391
Epoch 6/50, Loss: 1.7969
Epoch 7/50, Loss: 1.7647
Epoch 8/50, Loss: 1.7399
Epoch 9/50, Loss: 1.7206
Epoch 10/50, Loss: 1.7046
Epoch 11/50, Loss: 1.6907
Epoch 12/50, Loss: 1.6788
Epoch 13/50, Loss: 1.6689
Epoch 14/50, Loss: 1.6605
Epoch 15/50, Loss: 1.6531
Epoch 16/50, Loss: 1.6466
Epoch 17/50, Loss: 1.6409
Epoch 18/50, Loss: 1.6357
Epoch 19/50, Loss: 1.6309
Epoch 20/50, Loss: 1.6264
Epoch 21/50, Loss: 1.6222
Epoch 22/50, Loss: 1.6184
Epoch 23/50, Loss: 1.6148
Epoch 24/50, Loss: 1.6113
Epoch 25/50, Loss: 1.6081
Epoch 26/50, Loss: 1.6049
Epoch 27/50, Loss: 1.6021
Epoch 28/50, Loss: 1.5994
Epoch 29/50, Loss: 1.5968
Epoch 30/50, Loss: 1.5950
Epoch 31/50, Loss: 1.5930
Epoch 32/50, Loss: 1.5914
Epoch 33/50, Loss: 1.5901
Epoch 34/50, Loss: 1.5885
Epoch 35/50, Loss: 1.5863
Epoch 36/50, Loss: 1.5848
Epoch 37/50, Loss: 1.5827
Epoch 38/50, Loss

In [21]:
print(f"\nTraining RNN with 500 epochs:")
model = CharRNN(input_size=vocab_size, hidden_size=hidden_size, output_size=vocab_size)
model.char_to_idx = char_to_idx
model.idx_to_char = idx_to_char
model.chars = chars
model.to(device)
train_and_generate(model, encoded_text, vocab_size, seq_length, learning_rate, batch_size, num_epochs=500, start_text='The')



Training RNN with 500 epochs:
Epoch 1/500, Loss: 2.6962
Epoch 2/500, Loss: 2.1548
Epoch 3/500, Loss: 2.0118
Epoch 4/500, Loss: 1.9209
Epoch 5/500, Loss: 1.8592
Epoch 6/500, Loss: 1.8131
Epoch 7/500, Loss: 1.7785
Epoch 8/500, Loss: 1.7521
Epoch 9/500, Loss: 1.7319
Epoch 10/500, Loss: 1.7150
Epoch 11/500, Loss: 1.7002
Epoch 12/500, Loss: 1.6872
Epoch 13/500, Loss: 1.6761
Epoch 14/500, Loss: 1.6662
Epoch 15/500, Loss: 1.6576
Epoch 16/500, Loss: 1.6501
Epoch 17/500, Loss: 1.6435
Epoch 18/500, Loss: 1.6375
Epoch 19/500, Loss: 1.6321
Epoch 20/500, Loss: 1.6271
Epoch 21/500, Loss: 1.6226
Epoch 22/500, Loss: 1.6185
Epoch 23/500, Loss: 1.6145
Epoch 24/500, Loss: 1.6108
Epoch 25/500, Loss: 1.6074
Epoch 26/500, Loss: 1.6042
Epoch 27/500, Loss: 1.6014
Epoch 28/500, Loss: 1.5985
Epoch 29/500, Loss: 1.5963
Epoch 30/500, Loss: 1.5939
Epoch 31/500, Loss: 1.5922
Epoch 32/500, Loss: 1.5906
Epoch 33/500, Loss: 1.5887
Epoch 34/500, Loss: 1.5872
Epoch 35/500, Loss: 1.5855
Epoch 36/500, Loss: 1.5840
Epoch 

In [27]:

# LSTM model
class CharLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(CharLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x, hidden):
        out, hidden = self.lstm(x, hidden)
        out = self.fc(out)
        return out, hidden

    def init_hidden(self, batch_size):
        # LSTM has two hidden states: hidden state (h0) and cell state (c0)
        h0 = torch.zeros(1, batch_size, self.hidden_size, device=device)
        c0 = torch.zeros(1, batch_size, self.hidden_size, device=device)
        return (h0, c0)

print(f"\nTraining LSTM with 5 epochs:")
model = CharLSTM(input_size=vocab_size, hidden_size=hidden_size, output_size=vocab_size)
model.char_to_idx = char_to_idx
model.idx_to_char = idx_to_char
model.chars = chars
model.to(device)
train_and_generate(model, encoded_text, vocab_size, seq_length, learning_rate, batch_size, num_epochs=5, start_text='The')


Training LSTM with 5 epochs:
Epoch 1/5, Loss: 2.7091
Epoch 2/5, Loss: 2.1185
Epoch 3/5, Loss: 1.9582
Epoch 4/5, Loss: 1.8558
Epoch 5/5, Loss: 1.7832

Generated text after 5 epochs:
TheAere a duttent thee, that he shall
We can the read, shall the dearn thy suching, we come
That why wear sir as a dong the dauth
Beant a mence and then all be thy bring in the daughter son a blown the can 


In [28]:
print(f"\nTraining LSTM with 50 epochs:")
model = CharLSTM(input_size=vocab_size, hidden_size=hidden_size, output_size=vocab_size)
model.char_to_idx = char_to_idx
model.idx_to_char = idx_to_char
model.chars = chars
model.to(device)
train_and_generate(model, encoded_text, vocab_size, seq_length, learning_rate, batch_size, num_epochs=50, start_text='The')


Training LSTM with 50 epochs:
Epoch 1/50, Loss: 2.7150
Epoch 2/50, Loss: 2.1419
Epoch 3/50, Loss: 1.9887
Epoch 4/50, Loss: 1.8878
Epoch 5/50, Loss: 1.8088
Epoch 6/50, Loss: 1.7472
Epoch 7/50, Loss: 1.6998
Epoch 8/50, Loss: 1.6619
Epoch 9/50, Loss: 1.6305
Epoch 10/50, Loss: 1.6041
Epoch 11/50, Loss: 1.5820
Epoch 12/50, Loss: 1.5638
Epoch 13/50, Loss: 1.5471
Epoch 14/50, Loss: 1.5324
Epoch 15/50, Loss: 1.5197
Epoch 16/50, Loss: 1.5093
Epoch 17/50, Loss: 1.4997
Epoch 18/50, Loss: 1.4908
Epoch 19/50, Loss: 1.4829
Epoch 20/50, Loss: 1.4754
Epoch 21/50, Loss: 1.4684
Epoch 22/50, Loss: 1.4622
Epoch 23/50, Loss: 1.4571
Epoch 24/50, Loss: 1.4519
Epoch 25/50, Loss: 1.4470
Epoch 26/50, Loss: 1.4425
Epoch 27/50, Loss: 1.4385
Epoch 28/50, Loss: 1.4348
Epoch 29/50, Loss: 1.4312
Epoch 30/50, Loss: 1.4280
Epoch 31/50, Loss: 1.4248
Epoch 32/50, Loss: 1.4219
Epoch 33/50, Loss: 1.4197
Epoch 34/50, Loss: 1.4174
Epoch 35/50, Loss: 1.4142
Epoch 36/50, Loss: 1.4112
Epoch 37/50, Loss: 1.4095
Epoch 38/50, Los

In [29]:
print(f"\nTraining LSTM with 500 epochs:")
model = CharLSTM(input_size=vocab_size, hidden_size=hidden_size, output_size=vocab_size)
model.char_to_idx = char_to_idx
model.idx_to_char = idx_to_char
model.chars = chars
model.to(device)
train_and_generate(model, encoded_text, vocab_size, seq_length, learning_rate, batch_size, num_epochs=500, start_text='The')


Training LSTM with 500 epochs:
Epoch 1/500, Loss: 2.7125
Epoch 2/500, Loss: 2.1315
Epoch 3/500, Loss: 1.9733
Epoch 4/500, Loss: 1.8731
Epoch 5/500, Loss: 1.7998
Epoch 6/500, Loss: 1.7427
Epoch 7/500, Loss: 1.6949
Epoch 8/500, Loss: 1.6557
Epoch 9/500, Loss: 1.6234
Epoch 10/500, Loss: 1.5966
Epoch 11/500, Loss: 1.5737
Epoch 12/500, Loss: 1.5545
Epoch 13/500, Loss: 1.5379
Epoch 14/500, Loss: 1.5236
Epoch 15/500, Loss: 1.5113
Epoch 16/500, Loss: 1.5004
Epoch 17/500, Loss: 1.4908
Epoch 18/500, Loss: 1.4822
Epoch 19/500, Loss: 1.4747
Epoch 20/500, Loss: 1.4674
Epoch 21/500, Loss: 1.4608
Epoch 22/500, Loss: 1.4546
Epoch 23/500, Loss: 1.4486
Epoch 24/500, Loss: 1.4436
Epoch 25/500, Loss: 1.4390
Epoch 26/500, Loss: 1.4346
Epoch 27/500, Loss: 1.4310
Epoch 28/500, Loss: 1.4270
Epoch 29/500, Loss: 1.4233
Epoch 30/500, Loss: 1.4202
Epoch 31/500, Loss: 1.4175
Epoch 32/500, Loss: 1.4154
Epoch 33/500, Loss: 1.4134
Epoch 34/500, Loss: 1.4114
Epoch 35/500, Loss: 1.4094
Epoch 36/500, Loss: 1.4067
Epoch

###Task 2 (50 points):
In this task, you will pick a dataset (time-series or any other form of sequential data) and an associated problem that can be solved via sequence models. You must describe why you need sequence models to solve this problem. Include a link to the dataset source. Next, you should pick an RNN framework that you would use to solve this problem (This framework can be in TensorFlow, PyTorch or any other Python Package).

####Part 1 (10 points):
Implement your RNN either using an existing framework OR you can implement your own RNN cell structure. In either case, describe the structure of your RNN and the activation functions you are using for each time step and in the output layer. Define a metric you will use to measure the performance of your model (NOTE: Performance should be measured both for the validation set and the test set).

###Answer


####Part 2 (30 points):
Update your network from part 1 with first an LSTM and then a GRU
based cell structure (You can treat both as 2 separate implementations). Re-do the
training and performance evaluation. What are the major differences you notice? Why
do you think those differences exist between the 3 implementations (basic RNN, LSTM
and GRU)?

####Part 3 (10 points):
Can you use the traditional feed-forward network to solve the same
problem. Why or why not? (Hint: Can time series data be converted to usual features
that can be used as input to a feed-forward network?)

###Task 3 (50 points):

####Part 1: Implementing Word Embeddings (10 points)
- Use a pre-trained word embedding model (Word2Vec, GloVe, FastText, or BERT
embeddings).
- Provide a comparative discussion on why you chose this embedding over others.
- Load embeddings efficiently (either from pre-trained vectors or using an NLP library like
Gensim, SpaCy, or Hugging Face).
- Allow dynamic user input of two words and output their respective embeddings.
- Handle cases where a word is out of vocabulary (OOV) and suggest ways to approximate
its embedding.

####Part 2: Cosine Similarity Computation (20 points)
- Implement a function that computes the cosine similarity between two-word
embeddings.
- Explain why cosine similarity is useful in word embedding space.
- Allow batch processing, where users can input multiple word pairs for simultaneous
similarity computation.
- Visualization Requirement: Create a 2D or 3D scatter plot (e.g., using PCA or t-SNE) to
visually show how similar and dissimilar words cluster together in the embedding space.

####Part 3: Designing a Novel Dissimilarity Metric (20 points)
- Define a custom dissimilarity score that goes beyond cosine similarity. Possible
approaches include:
  - Euclidean distance (How far apart words are in vector space).
  - Word entropy-based dissimilarity (How uncommon two words are relative to
  each other in corpora).
  - Semantic contrast measure (Using external knowledge bases like WordNet).
- Either design your own metric or cite an existing one from literature (provide a proper reference).
  Explain why your metric captures novelty/diversity better than cosine
  similarity alone.
- Allow users to toggle between different similarity/dissimilarity measures via function
parameters.
- Visualization Requirement:
  - Plot the ranking of words based on their similarity/dissimilarity to a given word
  (e.g., how words like "cat" rank against "dog," "lion," and "table" using different
  metrics).
  - Use a heatmap to demonstrate and compare similarity and dissimilarity across
  multiple (any number of your choice) word pairs.